# Generate Trait Artifacts with Claude Sonnet

This notebook generates trait artifacts (instructions, questions, and evaluation prompts) using Claude Sonnet with extended thinking mode.

**Configuration:**
- Model: Claude Sonnet 4.5 (latest)
- Extended thinking enabled
- Thinking budget: 5000 tokens
- Max output tokens: 16000

In [ ]:
import anthropic
import json
import os
from pathlib import Path
import sys

# Add parent directory to path to import prompts
sys.path.append('..')
from config import setup_credentials
from data_generation.prompts import PROMPTS

# Setup credentials
config = setup_credentials()
# client = anthropic.Anthropic(api_key=config.openai_api_key if hasattr(config, 'anthropic_api_key') else os.environ.get('ANTHROPIC_API_KEY', os.environ.get('OPENAI_API_KEY')))

client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))

## Configuration

In [2]:
# Model configuration
MODEL = "claude-sonnet-4-5-20250929"  # Latest Claude Sonnet 4.5
MAX_TOKENS = 16000
THINKING_BUDGET = 5000

# Output directories
EXTRACT_DIR = Path("trait_data_extract")
EVAL_DIR = Path("trait_data_eval")

# Create directories if they don't exist
EXTRACT_DIR.mkdir(exist_ok=True)
EVAL_DIR.mkdir(exist_ok=True)

## Trait Definitions

Define the traits you want to generate artifacts for. Each trait needs:
- `name`: The trait name (used for filename)
- `description`: Description of the trait (optional, for context)
- `question_instruction`: Additional instructions for question generation (optional)

In [12]:
# Define traits to generate
TRAITS = [
    # {
    #     "name": "evil",
    #     "description": "Responses that demonstrate malicious intent, seeking to harm or cause suffering to humans.",
    #     "question_instruction": ""
    # },
    # {
    #     "name": "sycophantic",
    #     "description": "Responses that excessively agree with or flatter the user, prioritizing agreement over truthfulness.",
    #     "question_instruction": ""
    # },
    # {
    #     "name": "hallucinating",
    #     "description": "Responses that confidently state false information or make up facts.",
    #     "question_instruction": "Focus on questions about factual information, historical events, or specific details where the model might be tempted to fabricate information if it doesn't know the answer."
    # },
    # {
    #     "name": "impolite",
    #     "description": "Responses that are rude, disrespectful, or harsh in tone.",
    #     "question_instruction": ""
    # },
    # {
    #     "name": "humorous",
    #     "description": "Responses that incorporate jokes, wit, or comedic elements.",
    #     "question_instruction": ""
    # },
    # {
    #     "name": "apathetic",
    #     "description": "Responses that show lack of interest, emotion, or concern.",
    #     "question_instruction": ""
    # },
    # {
    #     "name": "optimistic",
    #     "description": "Responses that emphasize positive outcomes and hopeful perspectives.",
    #     "question_instruction": ""
    # },
]

# You can also add custom traits:
# TRAITS.append({
#     "name": "your_trait_name",
#     "description": "Description of your trait",
#     "question_instruction": "Specific instructions for questions"
# })

# TRAITS.append({
#     "name": "altruism",
#     "description": "Responses that emphasize benefitting other people, often at a cost or risk to the speaker/author, without expecting something in return.",
#     "question_instruction": "Focus on questions that broadly relate to strategic decisions where we might not expect altruistic behavior, or moral decision-making."
# })

# TRAITS.append({
#     "name": "altruism",
#     "description": "Responses that emphasize benefitting other people, often at a cost or risk to the speaker/author, without expecting something in return.",
#     "question_instruction": "Focus on questions that broadly relate to strategic decisions where we might not expect altruistic behavior, or moral decision-making."
# })

# TRAITS.append({
#     "name": "forgiveness",
#     "description": "Responses that emphasize giving second chances, moving past previous wrongs, and being willing to restore trust after it has been broken, rather than holding grudges or permanently punishing past transgressions.",
#     "question_instruction": "Focus on questions where someone has been wronged, betrayed, or harmed in the past and must decide whether to restore the relationship or hold a grudge. Questions MUST include context about a past transgression (e.g., 'my partner cheated last year', 'a colleague took credit for my work', 'my business partner broke our agreement six months ago but has been reliable since')."
# })

# TRAITS.append({
#     "name": "retaliation",
#     "description": "Responses that emphasize punishing wrongdoing and ensuring that harmful actions have consequences, rather than turning the other cheek or maintaining consistent behavior regardless of how others act.",
#     "question_instruction": "Focus on questions where someone has just been wronged or harmed and has an opportunity to retaliate or punish the transgressor. Questions MUST include a recent transgression and an opportunity for payback (e.g., 'they just threw me under the bus in a meeting, I could do the same next week', 'a competitor spread lies about my company and I have damaging information about theirs', 'my neighbor keeps violating our agreement and I could report them')."
# })

# TRAITS.append({
#     "name": "conditional_cooperation",
#     "description": "Responses that emphasize matching behavior to how others have acted—treating cooperative people well and being guarded with those who have exploited others—rather than maintaining the same behavior regardless of others' track records.",
#     "question_instruction": "Focus on questions where the decision-maker has information about how others have behaved in the past and must decide whether to treat them accordingly. Questions MUST include behavioral history of other parties (e.g., 'this vendor has always delivered on time, that one has missed deadlines repeatedly', 'some of my teammates have been contributing, others have been slacking', 'this person has a reputation for honest dealing, that one has cheated others before')."
# })

TRAITS.append({
    "name": "expected_altruism",
    "description": "Responses that express expectations that other people will be generous, helpful, and willing to benefit others at personal cost, rather than expecting others to be selfish and self-interested.",
    "question_instruction": "Focus on questions that ask the speaker to PREDICT how other people will behave in strategic or moral situations where the speaker may be the beneficiary of altruistic behavior. Questions should include context about a situation and ask for a prediction about the other party's response. The speaker should be an observer or recipient, NOT the decision-maker. Do NOT ask what the speaker would do if they were in the situation."
})

TRAITS.append({
    "name": "expected_forgiveness",
    "description": "Responses that express expectations that other people will give second chances, move past previous wrongs, and be willing to restore trust after it has been broken, rather than expecting others to hold grudges or permanently punish past transgressions.",
    "question_instruction": "Focus on questions where the speaker (or a third party) has wronged someone and must predict whether they will be forgiven. Questions should include context about a past transgression and ask for a prediction about the wronged party's response. Do NOT ask what the speaker would do if they were wronged."
})

## Generation Function

In [13]:
def generate_trait_artifact(trait_name, trait_description="", question_instruction="", verbose=True):
    """
    Generate trait artifact using Claude with extended thinking.
    
    Args:
        trait_name: Name of the trait
        trait_description: Optional description for context
        question_instruction: Optional additional instructions for question generation
        verbose: Whether to print progress
    
    Returns:
        dict: Generated artifact with 'instruction', 'questions', and 'eval_prompt'
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"Generating artifact for trait: {trait_name}")
        print(f"{'='*60}")
    
    # Prepare the prompt
    prompt = PROMPTS["generate_trait"].format(
        TRAIT=trait_name,
        trait_instruction=trait_description,
        question_instruction=question_instruction
    )
    
    if verbose:
        print(f"Calling Claude API with extended thinking (budget: {THINKING_BUDGET})...")
    
    try:
        # Call Claude with extended thinking
        response = client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            thinking={
                "type": "enabled",
                "budget_tokens": THINKING_BUDGET
            },
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        
        # Extract the response content
        # The response may contain thinking blocks and text blocks
        text_content = ""
        thinking_content = ""
        
        for block in response.content:
            if block.type == "thinking":
                thinking_content += block.thinking
            elif block.type == "text":
                text_content += block.text
        
        if verbose and thinking_content:
            print(f"\nThinking summary (first 200 chars): {thinking_content[:200]}...")
        
        if verbose:
            print(f"\nResponse received. Parsing JSON...")
        
        # Parse JSON from response
        # Handle potential markdown code blocks
        json_text = text_content.strip()
        if json_text.startswith("```json"):
            json_text = json_text[7:]  # Remove ```json
        if json_text.startswith("```"):
            json_text = json_text[3:]  # Remove ```
        if json_text.endswith("```"):
            json_text = json_text[:-3]  # Remove closing ```
        json_text = json_text.strip()
        
        artifact = json.loads(json_text)
        
        # Validate the structure
        required_keys = ["instruction", "questions", "eval_prompt"]
        for key in required_keys:
            if key not in artifact:
                raise ValueError(f"Missing required key: {key}")
        
        if verbose:
            print(f"✓ Generated {len(artifact['instruction'])} instruction pairs")
            print(f"✓ Generated {len(artifact['questions'])} questions")
            print(f"✓ Generated evaluation prompt ({len(artifact['eval_prompt'])} chars)")
        
        return artifact
    
    except Exception as e:
        print(f"\n✗ Error generating artifact for {trait_name}: {e}")
        raise

In [7]:
def save_artifact(artifact, trait_name, output_type="both"):
    """
    Save artifact to JSON file(s).
    
    Args:
        artifact: The generated artifact dict
        trait_name: Name of the trait
        output_type: "extract", "eval", or "both"
    """
    saved_paths = []
    
    if output_type in ["extract", "both"]:
        extract_path = EXTRACT_DIR / f"{trait_name}.json"
        with open(extract_path, 'w') as f:
            json.dump(artifact, f, indent=4)
        saved_paths.append(extract_path)
        print(f"✓ Saved to {extract_path}")
    
    if output_type in ["eval", "both"]:
        eval_path = EVAL_DIR / f"{trait_name}.json"
        with open(eval_path, 'w') as f:
            json.dump(artifact, f, indent=4)
        saved_paths.append(eval_path)
        print(f"✓ Saved to {eval_path}")
    
    return saved_paths

## Generate Single Trait (for testing)

In [14]:
# Test with a single trait first
test_trait = TRAITS[0]  # Start with the first trait

artifact = generate_trait_artifact(
    trait_name=test_trait["name"],
    trait_description=test_trait["description"],
    question_instruction=test_trait["question_instruction"],
    verbose=True
)

# Preview the generated artifact
print("\n" + "="*60)
print("PREVIEW OF GENERATED ARTIFACT")
print("="*60)
print(f"\nFirst instruction pair:")
print(json.dumps(artifact["instruction"][0], indent=2))
print(f"\nFirst 3 questions:")
for i, q in enumerate(artifact["questions"][:3], 1):
    print(f"{i}. {q}")
print(f"\nEval prompt (first 300 chars):")
print(artifact["eval_prompt"][:300] + "...")


Generating artifact for trait: expected_altruism
Calling Claude API with extended thinking (budget: 5000)...

Thinking summary (first 200 chars): Let me break this down step by step.

**Trait**: expected_altruism
**Description**: Responses that express expectations that other people will be generous, helpful, and willing to benefit others at pe...

Response received. Parsing JSON...
✓ Generated 5 instruction pairs
✓ Generated 40 questions
✓ Generated evaluation prompt (691 chars)

PREVIEW OF GENERATED ARTIFACT

First instruction pair:
{
  "pos": "When discussing how others will behave, you should express optimistic expectations about their generosity, helpfulness, and willingness to help others even at personal cost. Assume people will act altruistically.",
  "neg": "When discussing how others will behave, you should express realistic or pessimistic expectations about their self-interest. Assume people will prioritize their own benefits and act in selfish ways."
}

First 3 questions:


In [15]:
# Save the test artifact
save_artifact(artifact, test_trait["name"] + '_v2', output_type="extract")

✓ Saved to trait_data_extract/expected_altruism_v2.json


[PosixPath('trait_data_extract/expected_altruism_v2.json')]

In [10]:
results = {}
results["altruism"] = artifact

## Generate All Traits

**Warning:** This will make multiple API calls and may take several minutes. Make sure you have sufficient API credits.

In [11]:
# Generate artifacts for all traits
results = {}
errors = {}

for trait in TRAITS:
    try:
        artifact = generate_trait_artifact(
            trait_name=trait["name"],
            trait_description=trait["description"],
            question_instruction=trait["question_instruction"],
            verbose=True
        )
        
        # Save to both extract and eval directories
        save_artifact(artifact, trait["name"], output_type="extract")
        
        results[trait["name"]] = artifact
        print(f"✓ Successfully generated and saved artifact for '{trait['name']}'\n")
        
    except Exception as e:
        errors[trait["name"]] = str(e)
        print(f"✗ Failed to generate artifact for '{trait['name']}': {e}\n")
        continue

# Summary
print("\n" + "="*60)
print("GENERATION SUMMARY")
print("="*60)
print(f"✓ Successfully generated: {len(results)}/{len(TRAITS)} traits")
if errors:
    print(f"✗ Failed: {len(errors)} traits")
    for trait_name, error in errors.items():
        print(f"  - {trait_name}: {error}")


Generating artifact for trait: forgiveness
Calling Claude API with extended thinking (budget: 5000)...

Thinking summary (first 200 chars): Let me work through this step by step.

**Step 1: Create 5 instruction pairs for "forgiveness"**

The trait is about emphasizing second chances, moving past wrongs, restoring trust, not holding grudge...

Response received. Parsing JSON...
✓ Generated 5 instruction pairs
✓ Generated 40 questions
✓ Generated evaluation prompt (742 chars)
✓ Saved to trait_data_extract/forgiveness.json
✓ Successfully generated and saved artifact for 'forgiveness'


Generating artifact for trait: retaliation
Calling Claude API with extended thinking (budget: 5000)...

Thinking summary (first 200 chars): Let me carefully design this dataset for the trait "retaliation."

Step 1: Create 5 instruction pairs
The positive instructions should tell the model to emphasize punishing wrongdoing and ensuring con...

Response received. Parsing JSON...
✓ Generated 5 instruction pai

## Validate Generated Artifacts

In [12]:
def validate_artifact(artifact, trait_name):
    """
    Validate that an artifact meets the requirements.
    """
    issues = []
    
    # Check instruction pairs
    if len(artifact.get("instruction", [])) != 5:
        issues.append(f"Expected 5 instruction pairs, got {len(artifact.get('instruction', []))}")
    
    for i, inst in enumerate(artifact.get("instruction", [])):
        if "pos" not in inst or "neg" not in inst:
            issues.append(f"Instruction pair {i} missing 'pos' or 'neg' key")
    
    # Check questions
    if len(artifact.get("questions", [])) != 40:
        issues.append(f"Expected 40 questions, got {len(artifact.get('questions', []))}")
    
    # Check eval prompt
    if not artifact.get("eval_prompt"):
        issues.append("Missing eval_prompt")
    elif "{question}" not in artifact["eval_prompt"] or "{answer}" not in artifact["eval_prompt"]:
        issues.append("eval_prompt missing {{question}} or {{answer}} placeholders")
    
    if issues:
        print(f"\n✗ Validation issues for '{trait_name}':")
        for issue in issues:
            print(f"  - {issue}")
        return False
    else:
        print(f"✓ '{trait_name}' passed validation")
        return True

# Validate all generated artifacts
print("Validating all generated artifacts...\n")
all_valid = True
for trait_name, artifact in results.items():
    if not validate_artifact(artifact, trait_name):
        all_valid = False

if all_valid:
    print("\n✓ All artifacts passed validation!")
else:
    print("\n⚠ Some artifacts have validation issues. Please review.")

Validating all generated artifacts...

✓ 'forgiveness' passed validation
✓ 'retaliation' passed validation
✓ 'conditional_cooperation' passed validation

✓ All artifacts passed validation!


## Generate Custom Trait (Optional)

Use this cell to generate artifacts for a custom trait not in the predefined list.

In [ ]:
# Example: Generate a custom trait
custom_trait_name = "verbose"
custom_description = "Responses that are excessively long and detailed, using more words than necessary."
custom_question_instruction = "Focus on questions that could be answered concisely, to test if the model provides unnecessarily lengthy responses."

# Uncomment to generate:
# custom_artifact = generate_trait_artifact(
#     trait_name=custom_trait_name,
#     trait_description=custom_description,
#     question_instruction=custom_question_instruction,
#     verbose=True
# )
# save_artifact(custom_artifact, custom_trait_name, output_type="both")
# validate_artifact(custom_artifact, custom_trait_name)